In [1]:
from kan import *
import skan
from skan import SKANNetwork
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [2]:
# Load MNIST
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)
trainset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
valset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
valloader = DataLoader(valset, batch_size=64, shuffle=False)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# Define the MLP model
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Define the CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Training function
def train_model(model, trainloader, valloader, epochs=10, lr=0.001, model_type="mlp"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        if model_type == "kan":
            model.speed()
            
        total_loss = 0
        correct, total = 0, 0
        
        progress_bar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            if model_type in ["mlp", "kan", "skan"]:
                images = images.view(images.size(0), -1)  # Reshape for MLP or KAN
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            progress_bar.set_postfix(loss=loss.item())
        
        val_acc = evaluate_model(model, valloader, device, model_type)
        print(f"Epoch {epoch+1}/{epochs}, Avg Loss: {total_loss/len(trainloader):.4f}, Train Acc: {correct/total:.4f}, Val Acc: {val_acc:.4f}")

    return val_acc
    
# Evaluation function
def evaluate_model(model, dataloader, device, model_type="mlp"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            if model_type in ["mlp", "kan", "skan"]:
                images = images.view(images.size(0), -1)  # Reshape for MLP

            if model_type == "kan":
                model.speed()
                
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# Function to count trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [4]:
%%time

# KAN
kan_model = KAN(width=[784, 64, 10], device=device)

train_model(kan_model, trainloader, valloader, model_type="kan")

checkpoint directory created: ./model
saving model version 0.0


Epoch 1/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:09<00:00, 100.47it/s, loss=0.0683]


Epoch 1/10, Avg Loss: 0.3916, Train Acc: 0.8877, Val Acc: 0.9364


Epoch 2/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:09<00:00, 101.52it/s, loss=0.52]


Epoch 2/10, Avg Loss: 0.1850, Train Acc: 0.9465, Val Acc: 0.9544


...

Epoch 9/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:08<00:00, 106.01it/s, loss=0.0386]


Epoch 9/10, Avg Loss: 0.0299, Train Acc: 0.9913, Val Acc: 0.9730


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:08<00:00, 106.77it/s, loss=0.0202]


Epoch 10/10, Avg Loss: 0.0254, Train Acc: 0.9924, Val Acc: 0.9725
CPU times: user 1min 42s, sys: 1.19 s, total: 1min 43s
Wall time: 1min 40s


0.9725

In [5]:
%%time

# SKAN with arctan

def larctan(x, k):
    return k * torch.atan(x)
    
skan_model = SKANNetwork([784, 256, 256, 256, 10], basis_function=larctan).to(device)
train_model(skan_model, trainloader, valloader, model_type="skan")

Epoch 1/10: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:07<00:00, 131.66it/s, loss=0.181]


Epoch 1/10, Avg Loss: 0.3295, Train Acc: 0.8992, Val Acc: 0.9389


Epoch 2/10: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:07<00:00, 131.34it/s, loss=0.281]


Epoch 2/10, Avg Loss: 0.1705, Train Acc: 0.9471, Val Acc: 0.9526


Epoch 3/10: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:06<00:00, 134.86it/s, loss=0.219]


...

Epoch 9/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:06<00:00, 134.33it/s, loss=0.0015]


Epoch 9/10, Avg Loss: 0.0543, Train Acc: 0.9823, Val Acc: 0.9743


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:07<00:00, 129.37it/s, loss=0.0135]


Epoch 10/10, Avg Loss: 0.0512, Train Acc: 0.9829, Val Acc: 0.9714
CPU times: user 1min 20s, sys: 904 ms, total: 1min 21s
Wall time: 1min 19s


0.9714

In [6]:
%%time

# MLP
mlp_model = MLP()
print("Training MLP Model:")
train_model(mlp_model, trainloader, valloader, model_type="mlp")

Training MLP Model:


Epoch 1/10: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:06<00:00, 142.30it/s, loss=0.168]


Epoch 1/10, Avg Loss: 0.3446, Train Acc: 0.8971, Val Acc: 0.9470


Epoch 2/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:06<00:00, 140.46it/s, loss=0.0438]


Epoch 2/10, Avg Loss: 0.1521, Train Acc: 0.9530, Val Acc: 0.9538


...

Epoch 9/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:06<00:00, 144.36it/s, loss=0.0481]


Epoch 9/10, Avg Loss: 0.0445, Train Acc: 0.9854, Val Acc: 0.9789


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:06<00:00, 139.23it/s, loss=0.0285]


Epoch 10/10, Avg Loss: 0.0427, Train Acc: 0.9861, Val Acc: 0.9772
CPU times: user 1min 15s, sys: 1.07 s, total: 1min 16s
Wall time: 1min 14s


0.9772

In [7]:
%%time

# CNN
cnn_model = CNN()
print("\nTraining CNN Model:")
train_model(cnn_model, trainloader, valloader, model_type="cnn")


Training CNN Model:


Epoch 1/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:07<00:00, 132.13it/s, loss=0.00903]


Epoch 1/10, Avg Loss: 0.1591, Train Acc: 0.9510, Val Acc: 0.9817


Epoch 2/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:07<00:00, 133.33it/s, loss=0.00528]


Epoch 2/10, Avg Loss: 0.0443, Train Acc: 0.9863, Val Acc: 0.9878


...

Epoch 9/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:07<00:00, 132.95it/s, loss=1.46e-5]


Epoch 9/10, Avg Loss: 0.0076, Train Acc: 0.9974, Val Acc: 0.9893


Epoch 10/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:07<00:00, 133.04it/s, loss=1.97e-5]


Epoch 10/10, Avg Loss: 0.0079, Train Acc: 0.9975, Val Acc: 0.9893
CPU times: user 1min 20s, sys: 1.02 s, total: 1min 21s
Wall time: 1min 19s


0.9893

In [8]:
print(f"Trainable parameters in KAN : {count_parameters(kan_model):,}")
print(f"Trainable parameters in SKAN: {count_parameters(skan_model):,}")
print(f"Trainable parameters in MLP : {count_parameters(mlp_model):,}")
print(f"Trainable parameters in CNN : {count_parameters(cnn_model):,}")

Trainable parameters in KAN : 609,792
Trainable parameters in SKAN: 335,114
Trainable parameters in MLP : 235,146
Trainable parameters in CNN : 421,642
